# 42_04 - RQ1: Kerentanan FedAvg vs fraksi penyerang (Paper 4-2)**Tujuan (RQ1):** ukur penurunan MCC FedAvg seiring naiknya fraksi penyerangf in {0,10,20,30}% (0/1/2/3 klien jahat dari K=10) untuk tiga serangan(label-flip, scaling, backdoor). Bukti **kerentanan** FedAvg -> motivasi pertahanan.**Sumber tunggal:** `robust_agg.py` (serangan) + `../aws/fed_node.py` (MLP/FedAvg).**Output:** `fed_out42/vuln_fedavg.csv` (Tabel RQ1 di paper-4-2.tex).> Jalankan setelah 42_01. JANGAN mengarang angka.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.')); import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEED=42
print('setup 42_04 siap | LAYERS',fed.LAYERS)

## 1. Muat partisi + helper run_fed (reuse robust_agg + fed_node)

In [ ]:
def need(path):
    assert os.path.exists(path),'PRASYARAT GAGAL: %s - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')),allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')),allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
def predict_with(W,mu,sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5,
            rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy(); rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        if len(yk): mu,sd=fed.zscore_fit(Xk)
        else: mu,sd=np.zeros(X.shape[1]),np.ones(X.shape[1])
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd),y=yk,n=len(yk)))
    Wg=fed.init_weights(seed); f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(rounds):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg,cli[k]['X'],cli[k]['y'],epochs=epochs,lr=lr,mu=0.0,W_global=Wg,seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl,gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule,wl,sizes,f=f_byz,beta=max(1,f_byz))
    mu_g,sd_g=fed.zscore_fit(X); pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    mcc=round(matthews_corrcoef(yg,yp),4)
    asr=round(ra.attack_success_rate(pf,Xg,yg),4) if attack=='backdoor' else None
    return mcc,asr,Wg
print('K',K,'| pool',X.shape,'| run_fed siap')

## 2. Sweep fraksi penyerang x serangan (FedAvg)

In [ ]:
frac_map={0.0:(),0.1:(0,),0.2:(0,1),0.3:(0,1,2)}
rows=[]
for atk in ['label_flip','scale','backdoor']:
    for frac,mal in frac_map.items():
        attack=None if frac==0.0 else atk
        mcc,asr,_=run_fed('fedavg', attack=attack, mal_ids=mal)
        rows.append({'attack':atk,'frac':frac,'n_mal':len(mal),'MCC':mcc,'ASR':asr})
        print(f'{atk} f={int(frac*100)}% -> MCC {mcc} ASR {asr}')
dfv=pd.DataFrame(rows); dfv.to_csv(os.path.join(OUTDIR,'vuln_fedavg.csv'),index=False)
import IPython.display as ipd; ipd.display(dfv.pivot(index='attack',columns='frac',values='MCC'))
print('=== SEL 2 (sweep kerentanan) SELESAI ===')

## 3. Catatan untuk naskah- Hasil -> **Tabel RQ1** (`paper-4-2.tex`, tab:vuln). MCC FedAvg vs fraksi penyerang.- Diharapkan MCC turun saat f naik; ASR backdoor naik. Dilaporkan apa adanya.- f=0% = baseline bersih (acuan 'harga ketahanan' di 42_05).- Serangan & agregasi identik `robust_agg.py` -> konsisten dgn 42_05/42_06/EC2.